# Part 3 : Snowflake Intelligence

## このパートでやること

**Snowflake Intelligence（SI）** は、自然言語で Snowflake のデータに質問できる AI アシスタントです。

このパートでは SI を実際に使って、インフルエンサーの SNS 投稿データと売上データを分析してみます。

### ストーリー

ある EC ブランド「GlacierStyle」のデータアナリストになったつもりで進めましょう。

> 「最近インフルエンサーが GlacierStyle の商品を紹介してくれているんだけど、SNS のバズが売上にどれくらい影響しているか知りたいんです！」

このような業務的な質問を、SQL を書かずに **自然言語で Snowflake に問いかける** ことができます。

### このパートの流れ

| # | 内容 | 所要時間 |
|---|---|---|
| 1 | データ確認（SNS + 売上データ） | 5分 |
| 2 | Semantic View の作成 | 10分 |
| 3 | SI で質問してみよう（成功例） | 10分 |
| 4 | SI が答えられない質問をしてみよう（失敗例） | 10分 |
| 5 | Semantic View を改善する | 10分 |
| 6 | 再チャレンジ！ | 10分 |

**前提条件:** Part 1・Part 2 が完了していること

In [ ]:
-- 環境設定
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE GLACIERSTYLE_WH;
USE DATABASE GLACIERSTYLE_DB;
USE SCHEMA EC_ANALYTICS_SCHEMA;


## 1. データ確認

今回 SI で分析に使うデータを確認しておきましょう。

### 使用するテーブル

| テーブル | 内容 |
|---|---|
| `raw_sns_mentions` | インフルエンサーの SNS 投稿（いいね・RT 数など） |
| `fact_orders` | 注文データ（売上金額・日時など） |
| `dim_products` | 商品マスタ（商品名・カテゴリなど） |

In [ ]:
-- SNS 投稿データを確認
SELECT
    post_id,
    platform,
    username,
    display_name,
    posted_at,
    likes,
    retweets,
    replies,
    content
FROM raw_sns_mentions
LIMIT 5;


In [ ]:
-- 売上データを確認
SELECT
    order_id,
    order_datetime,
    product_id,
    quantity,
    total_amount
FROM fact_orders
LIMIT 5;


## 2. Semantic View を作成する

**Semantic View（セマンティックビュー）** は、SI が「このデータは何を意味するか」を理解するためのメタデータ定義です。

- テーブルの意味や関係性を定義する
- ビジネス指標（メトリクス）をあらかじめ定義しておく
- SI はこの定義をもとに自然言語クエリを SQL に変換する

### ポイント

Semantic View に **定義されていないメトリクスは SI が答えられません。**
後のセクションでこれを体験します。

### 今回作成する Semantic View

| 要素 | 内容 |
|---|---|
| ベーステーブル | `raw_sns_mentions`（SNS 投稿） |
| 関連テーブル | `fact_orders`（売上）, `dim_products`（商品） |
| ディメンション | 投稿プラットフォーム、ユーザー名、商品カテゴリ、投稿日付 |
| メトリクス | 合計売上、注文件数、投稿件数 |

In [ ]:
-- ========================================================
-- Semantic View の作成（ドラフト / TODO: 内容差し替え予定）
-- ========================================================
CREATE OR REPLACE SEMANTIC VIEW sv_influencer_analysis
  TABLES (
    raw_sns_mentions PRIMARY (
      DIMENSIONS (
        post_id          COMMENT '投稿ID',
        platform         COMMENT '投稿プラットフォーム（Instagram / X など）',
        username         COMMENT 'インフルエンサーのユーザー名',
        display_name     COMMENT '表示名',
        posted_at        COMMENT '投稿日時',
        hashtags         COMMENT 'ハッシュタグ'
      )
      METRICS (
        total_posts    AS COUNT(post_id)       COMMENT '総投稿件数',
        total_likes    AS SUM(likes)           COMMENT '合計いいね数',
        total_retweets AS SUM(retweets)        COMMENT '合計リツイート数',
        total_replies  AS SUM(replies)         COMMENT '合計リプライ数'
      )
    ),
    fact_orders (
      DIMENSIONS (
        order_id        COMMENT '注文ID',
        order_datetime  COMMENT '注文日時',
        product_id      COMMENT '商品ID',
        order_channel   COMMENT '注文チャネル'
      )
      METRICS (
        total_revenue   AS SUM(total_amount)   COMMENT '合計売上金額',
        total_orders    AS COUNT(order_id)     COMMENT '注文件数'
      )
    ),
    dim_products (
      DIMENSIONS (
        product_id    COMMENT '商品ID',
        product_name  COMMENT '商品名',
        category_l1   COMMENT '商品カテゴリ（大）',
        category_l2   COMMENT '商品カテゴリ（中）',
        brand         COMMENT 'ブランド名'
      )
    )
  )
  RELATIONSHIPS (
    fact_orders (product_id) REFERENCES dim_products (product_id)
    -- TODO: SNS 投稿と注文の関連付け方法を検討する
  );


In [ ]:
-- 作成した Semantic View を確認
DESCRIBE SEMANTIC VIEW sv_influencer_analysis;


## 3. Snowflake Intelligence で質問してみよう（成功例）

Semantic View の準備ができました。いよいよ SI を使ってみましょう。

### SI の起動方法

1. Snowsight の左メニューから **「Snowflake Intelligence」** をクリック
2. 先ほど作成した `sv_influencer_analysis` を選択
3. チャット欄に質問を入力

<!-- TODO: スクリーンショット画像を追加 -->
<!-- ![SI Launch](images/part3/01_si_launch.png) -->

### 試してみる質問（シンプルな集計）

まずは SI が確実に答えられる**シンプルな質問**を試してみましょう。

```
売上の高い商品カテゴリ上位5つを教えてください
```

```
先月の注文件数は何件ですか？
```

```
投稿数が最も多いインフルエンサーは誰ですか？
```

> **確認ポイント:** SI が SQL を自動生成して結果を返してくれることを確認しましょう。

In [ ]:
-- 参考: SI が生成する SQL と同等のクエリを手動で確認
-- SI の回答と結果が一致するか検証に使えます
SELECT
    p.category_l1,
    SUM(o.total_amount) AS total_revenue
FROM fact_orders o
JOIN dim_products p ON o.product_id = p.product_id
GROUP BY p.category_l1
ORDER BY total_revenue DESC
LIMIT 5;


## 4. SI が答えられない質問をしてみよう（失敗例）

今度は少し**踏み込んだ質問**を SI にしてみましょう。

### 試してみる質問

```
インフルエンサーのバズスコアと売上の相関を見せてください
```

```
バズった投稿の翌日の売上はどれくらい増えましたか？
```

> **予想:** SI はこれらの質問に答えられないか、的外れな回答をするはずです。

### なぜ答えられないのか？

**「バズスコア」という概念が Semantic View に定義されていないから**です。

- SI は Semantic View の定義をもとに SQL を組み立てます
- `buzz_score`（バズスコア）というメトリクスが存在しない
- SNS 投稿と売上の時系列的な関係も定義されていない

→ **Semantic View を改善して、このような質問にも答えられるようにしましょう！**

## 5. Semantic View を改善する

**バズスコア** のメトリクスを Semantic View に追加します。

### バズスコアの定義

バズスコアは以下のように定義します：

```
buzz_score = likes + retweets * 2 + replies
```

※ リツイートはリーチへの影響が大きいため 2 倍の重みを設定

### 改善するポイント

1. `buzz_score` メトリクスの追加
2. Verified Query（検証済みクエリ）の追加
   - SI が「バズスコア」と聞かれたとき、正しい SQL を使うためのヒント集

> **Verified Query とは？**
> よく聞かれる質問とその正解 SQL をペアで登録することで、SI の回答精度が向上します。

In [ ]:
-- ========================================================
-- Semantic View を改善版に再作成（buzz_score メトリクスを追加）
-- TODO: 内容差し替え予定
-- ========================================================
CREATE OR REPLACE SEMANTIC VIEW sv_influencer_analysis
  TABLES (
    raw_sns_mentions PRIMARY (
      DIMENSIONS (
        post_id          COMMENT '投稿ID',
        platform         COMMENT '投稿プラットフォーム（Instagram / X など）',
        username         COMMENT 'インフルエンサーのユーザー名',
        display_name     COMMENT '表示名',
        posted_at        COMMENT '投稿日時',
        hashtags         COMMENT 'ハッシュタグ'
      )
      METRICS (
        total_posts    AS COUNT(post_id)                          COMMENT '総投稿件数',
        total_likes    AS SUM(likes)                             COMMENT '合計いいね数',
        total_retweets AS SUM(retweets)                          COMMENT '合計リツイート数',
        total_replies  AS SUM(replies)                           COMMENT '合計リプライ数',
        buzz_score     AS SUM(likes + retweets * 2 + replies)    COMMENT 'バズスコア（いいね + RT×2 + リプライ）'
      )
    ),
    fact_orders (
      DIMENSIONS (
        order_id        COMMENT '注文ID',
        order_datetime  COMMENT '注文日時',
        product_id      COMMENT '商品ID',
        order_channel   COMMENT '注文チャネル'
      )
      METRICS (
        total_revenue   AS SUM(total_amount)   COMMENT '合計売上金額',
        total_orders    AS COUNT(order_id)     COMMENT '注文件数'
      )
    ),
    dim_products (
      DIMENSIONS (
        product_id    COMMENT '商品ID',
        product_name  COMMENT '商品名',
        category_l1   COMMENT '商品カテゴリ（大）',
        category_l2   COMMENT '商品カテゴリ（中）',
        brand         COMMENT 'ブランド名'
      )
    )
  )
  RELATIONSHIPS (
    fact_orders (product_id) REFERENCES dim_products (product_id)
  )
  VERIFIED QUERIES (
    -- TODO: Verified Query を追加する
    -- 例:
    --   question: 'インフルエンサーのバズスコアランキングを教えてください'
    --   sql: SELECT username, SUM(likes + retweets * 2 + replies) AS buzz_score
    --        FROM raw_sns_mentions GROUP BY username ORDER BY buzz_score DESC
  );


In [ ]:
-- バズスコアの計算結果を手動で確認（期待値チェック）
SELECT
    username,
    display_name,
    SUM(likes)                          AS total_likes,
    SUM(retweets)                       AS total_retweets,
    SUM(replies)                        AS total_replies,
    SUM(likes + retweets * 2 + replies) AS buzz_score
FROM raw_sns_mentions
GROUP BY username, display_name
ORDER BY buzz_score DESC
LIMIT 10;


## 6. 再チャレンジ！（改善後）

Semantic View を改善しました。もう一度 SI に同じ質問をしてみましょう。

### 再度試してみる質問

```
インフルエンサーのバズスコアランキングを見せてください
```

```
バズスコアが高いインフルエンサーはどんな商品カテゴリを多く紹介していますか？
```

```
カテゴリ別のバズスコアと売上の関係を教えてください
```

> **確認ポイント:**
> - 先ほど答えられなかった質問に、今度は正しく回答できるか確認しましょう
> - SI が自動生成した SQL を見て、期待通りの集計になっているか確認しましょう

### まとめ

| | 改善前 | 改善後 |
|---|---|---|
| バズスコアの質問 | ❌ 答えられない | ✅ 正しく回答 |
| シンプルな集計 | ✅ 正しく回答 | ✅ 正しく回答 |

**Semantic View の定義がそのまま SI の「答えられる範囲」になります。**
ビジネス要件が追加されるたびに Semantic View を育てていくことで、SI はどんどん賢くなります。

## お疲れ様でした！

### Part 3 で学んだこと

- **Snowflake Intelligence（SI）** を使って自然言語でデータに質問できた
- **Semantic View** の定義が SI の回答精度に直結することを体験した
- メトリクスの追加（バズスコア）によって SI の回答範囲を広げた

### 次のステップ

- Part 4: Snowflake Marketplace（外部データとの連携）